🎯 **You Sank My Function** — Black-Box Optimisation Capstone · Imperial College Business School

**Weekly pipeline** · [04 Consolidate](../pipeline/04_consolidate_data.ipynb) → **05b Suggest Engine v2** → [06 Diagnose](../pipeline/06_diagnose.ipynb)

> 📍 **This notebook** — the *generalized* suggestion engine. Instead of eight hand-written RECORDS, it
> (1) **classifies** each function into a pathology archetype from the data,
> (2) routes it to a **strategy module** that generates candidate probes,
> (3) prices every candidate under **one referee surrogate** (EI + P(beat best-ever)),
> (4) enforces the **global rules** (bounds, no-repeat ε-fix), and
> (5) builds the submission with full provenance and logged human overrides.

---

# 05b · Generalized Suggestion Engine
### Pathology → strategy → referee → constraints → submission

| archetype | signature in the data | strategy module | campaign example |
|---|---|---|---|
| `needle` | magnitude cascade over ≥6 decades, ≥50 % background | log-quadratic vertex fit near the peak + micro trust region | F1 |
| `stochastic_capped` | best-ever is a week-≤1 draw; nearby re-reads never re-attain it (winner's curse) | lottery: near-incumbent re-reads + maximin exploration, human picks EV-max vs P(beat)-max | F2, F3 |
| `local_peak` | new best-ever last week from a small step — a verified gradient | TuRBO-lite: continue last step ×1.5 on success / reverse-halve on failure + TR cloud | F4, (F6 after W7) |
| `boundary_climb` | incumbent pinned at bounds, strong monotone trends, huge range | coordinate isolation scans with **capped extrapolation**, priced on log scale | F5 |
| `plateau` | no new best for ≥3 weeks, not flat, not solved | price escape hypotheses: separable optimum, anti-corr pairs, single-factor probes | F6 (W4–W6) |
| `solved_lock` | analytic identity confirmed (human prior) | ε-perturbation on least-sensitive dim, novelty-checked | F7, F8 |

The classifier is deliberately **re-run every week**: archetypes are states, not identities.
F6 is the proof — plateau in weeks 4–6, `local_peak` the moment the single-factor probe landed a new best.

## 0 · Configuration & data

In [1]:
import warnings
import numpy as np, pandas as pd
from scipy.stats import norm, spearmanr
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.linear_model import Ridge
warnings.filterwarnings("ignore")

CONSOLIDATED = "consolidated_observations.csv"     # from 04_consolidate_data
SEED = 42
OBS  = pd.read_csv(CONSOLIDATED)
DIMS = OBS.groupby("function").dim.first().to_dict()
N_WEEKS = int(OBS.week.max())

# Human priors that data cannot supply: which functions are analytically solved.
LOCK_PRIOR = (7, 8)
# Optional per-function pathology overrides, e.g. {2: "plateau"}. Leave empty to trust the classifier.
PATHOLOGY_OVERRIDES = {}

def xy(fid, df=None):
    df = OBS if df is None else df
    sub = df[df.function == fid]; d = DIMS[fid]
    return sub[[f"x{i+1}" for i in range(d)]].to_numpy(float), sub["y"].to_numpy(float), sub

def weekly_trace(fid, df=None):
    _, _, sub = xy(fid, df)
    wk = sub[sub.week > 0].sort_values("week")
    return wk[[f"x{i+1}" for i in range(DIMS[fid])]].to_numpy(float), wk["y"].to_numpy(float)

print(f"Loaded {len(OBS)} observations · weeks 0–{N_WEEKS} · suggesting for week {N_WEEKS+1}")

Loaded 279 observations · weeks 0–13 · suggesting for week 14


## 1 · Pathology classifier

Cheap signals only (the Tier-0 philosophy from `06_diagnose`), resolved in a fixed priority order.
Every verdict carries an evidence string, and the whole thing can be overridden per function —
overrides are *logged*, not silent.

In [2]:
def sig_needle(y):
    ay = np.abs(y); eps = 1e-300
    decades = float(np.log10(ay.max()+eps) - np.log10(np.median(ay)+eps))
    background = float(np.mean(ay < 0.01*ay.max()))
    return decades, background

def sig_noise(fid, df=None):
    """Noise sd from near-exact resamples (dist<0.01) if any, else a GP WhiteKernel estimate."""
    X, y, _ = xy(fid, df); pairs = []
    for i in range(len(X)):
        for j in range(i+1, len(X)):
            if np.linalg.norm(X[i]-X[j]) < 0.01:
                pairs.append(abs(y[i]-y[j]))
    if pairs:
        return float(np.median(pairs)/np.sqrt(2)), "near-exact resamples"
    k = C(1.0)*Matern(0.2, nu=2.5) + WhiteKernel(1e-3, (1e-12, 1e2))
    gp = GaussianProcessRegressor(kernel=k, normalize_y=True,
                                  n_restarts_optimizer=2, random_state=SEED).fit(X, y)
    return float(np.sqrt(gp.kernel_.k2.noise_level)*np.std(y)), "GP WhiteKernel"

def classify(fid, df=None):
    """Priority: lock > needle > boundary_climb > stochastic_capped > local_peak (fresh gradient)
    > plateau > local_peak (default)."""
    if fid in PATHOLOGY_OVERRIDES:
        return PATHOLOGY_OVERRIDES[fid], ["HUMAN OVERRIDE of pathology"]
    if fid in LOCK_PRIOR:
        return "solved_lock", ["analytic identity confirmed (human prior); optimum banked"]
    X, y, sub = xy(fid, df); d = DIMS[fid]
    Xw, yw = weekly_trace(fid, df)

    decades, bg = sig_needle(y)
    if decades >= 6 and bg >= 0.5:
        return "needle", [f"magnitude cascade {decades:.0f} decades; {bg:.0%} background"]

    i_best = int(np.argmax(y)); best_week = int(sub.iloc[i_best]["week"])
    at_bound = int(np.sum((X[i_best] < .03) | (X[i_best] > .97)))
    rhos = [abs(spearmanr(X[:, j], y)[0] or 0) for j in range(d)]
    if at_bound >= d-1 and np.mean(rhos) > .4 and y.max() > 10*abs(np.median(y)):
        return "boundary_climb", [f"{at_bound}/{d} coords at bounds; mean |rho|={np.mean(rhos):.2f}; huge monotone range"]

    noise, src = sig_noise(fid, df)
    if noise > 0 and best_week <= 1:
        near = [yy for xx, yy in zip(X, y)
                if np.linalg.norm(xx - X[i_best]) < 0.08 and yy < y.max()]
        if len(near) >= 2 and max(near) < y.max() and np.mean(near) < y.max() - noise:
            return "stochastic_capped", [
                f"best {y.max():.4g} is a week-{best_week} draw; {len(near)} nearby re-reads "
                f"(mean {np.mean(near):.4g}) never re-attained it — winner's curse "
                f"(noise sd≈{noise:.3g} from {src})"]

    if len(yw) >= 2:
        prev_best = np.max(sub[sub.week < sub.week.max()].y.values)
        step = np.linalg.norm(Xw[-1] - Xw[-2])
        if yw[-1] >= prev_best - 1e-12 and step < 0.25:
            return "local_peak", [f"new best-ever last week (+{yw[-1]-prev_best:.3g}) "
                                  f"via a {step:.3f}-norm step — verified local gradient"]
        weeks_since = int(sub.week.max() - best_week)
        if weeks_since >= 3:
            return "plateau", [f"no new best for {weeks_since} weeks; incumbent {y.max():.4g}"]
    return "local_peak", ["default: learnable, not stalled"]

## 2 · The referee — one surrogate prices every ticket

The single criterion that converts strategy scatter back into one decision (the campaign's
"ensemble accident that became a method"). Every candidate, from every module, gets **EI** and
**P(beat best-ever)** under the *same* noise-aware GP. Rank by EI by default; the P(beat) column
is what a risk-averse human overrides on (F3, week 7).

In [3]:
def fit_ref(fid, log=False, df=None):
    X, y, _ = xy(fid, df)
    ym = np.log(y - y.min() + 1e-3*(y.max()-y.min()) + 1e-12) if log else y
    k = C(1.0)*Matern(0.2, nu=2.5, length_scale_bounds=(0.02, 2.0)) + WhiteKernel(1e-4, (1e-12, 1e2))
    gp = GaussianProcessRegressor(kernel=k, normalize_y=True,
                                  n_restarts_optimizer=3, random_state=SEED).fit(X, ym)
    return gp, ym

def price(gp, ym_best, XC):
    mu, s = gp.predict(np.atleast_2d(XC), return_std=True)
    s = np.maximum(s, 1e-12)
    z = (mu - ym_best) / s
    ei = (mu - ym_best)*norm.cdf(z) + s*norm.pdf(z)
    return ei, norm.cdf(z), mu, s

## 3 · Global constraints — the F8 save, institutionalized

Every final pick passes through the no-repeat guard: if it collides (to 6 dp, the portal's
resolution) with **any** past query, it gets an ε-nudge on the least-sensitive dimension until novel.
This is the automated version of the trailing-digit catch that saved the week-7 F8 probe.

In [4]:
def past_set(fid, dp=6, df=None):
    X, _, _ = xy(fid, df)
    return {tuple(np.round(x, dp)) for x in X}

def is_repeat(fid, x, dp=6, df=None):
    return tuple(np.round(np.asarray(x, float), dp)) in past_set(fid, dp, df)

def epsilon_fix(fid, x, eps=1e-6, df=None):
    x = np.asarray(x, float).copy()
    if not is_repeat(fid, x, df=df):
        return x, None
    X, y, _ = xy(fid, df); d = DIMS[fid]
    order = np.argsort([abs(spearmanr(X[:, j], y)[0] or 0) for j in range(d)])  # least-sensitive first
    for j in order:
        for sgn in (+1, -1):
            c = x.copy(); c[j] = np.clip(c[j] + sgn*eps, 0, 1)
            if not is_repeat(fid, c, df=df):
                return c, f"repeat detected -> eps {sgn*eps:+g} on x{j+1} (least-sensitive)"
    raise RuntimeError("no novel epsilon found — widen eps")

## 4 · Strategy library

Each module returns a **list of candidates with provenance strings** — never a single opaque pick.
The referee ranks them; the table below each function shows why the winner won.

In [5]:
def maximin_candidates(fid, n=3, pool=6000, rng=None, df=None):
    """The 'emptiest part of the map' probes (F2/F3 exploration tickets)."""
    rng = rng or np.random.default_rng(SEED + fid)
    X, _, _ = xy(fid, df); d = DIMS[fid]
    P = rng.uniform(0, 1, (pool, d)); out = []; Xa = X.copy()
    for _ in range(n):
        dmin = np.min(np.linalg.norm(P[:, None, :] - Xa[None, :, :], axis=2), axis=1)
        i = int(np.argmax(dmin))
        out.append((P[i], f"maximin explore (nearest sample {dmin[i]:.2f})"))
        Xa = np.vstack([Xa, P[i]]); P = np.delete(P, i, 0)
    return out

def local_cloud(fid, r=0.05, n=300, rng=None, df=None):
    rng = rng or np.random.default_rng(SEED + fid)
    X, y, _ = xy(fid, df)
    inc = X[np.argmax(y)]
    return np.clip(inc + rng.normal(0, r, (n, DIMS[fid])), 0, 1)

# --- needle (F1): log-quadratic vertex on the NEAR-PEAK points only -------------
def strat_needle(fid, df=None):
    X, y, _ = xy(fid, df); out = []
    inc = X[np.argmax(y)]
    pos = np.where(y > 0)[0]
    lt_all = np.log(y[pos] + 1e-300)
    sel = pos[lt_all >= lt_all.max() - 5]          # within 5 log-units of the peak (~x150)
    if len(sel) >= 3:
        Xt, lt = X[sel], np.log(y[sel])
        w = lt - lt.min() + 1e-9; w /= w.sum()      # weight toward the peak
        Xc = (Xt - (Xt*w[:, None]).sum(0)) * np.sqrt(w)[:, None]
        u = np.linalg.svd(Xc, full_matrices=False)[2][0]     # dominant line
        t = (Xt - inc) @ u
        A = np.column_stack([np.ones_like(t), t, t**2])
        coef, *_ = np.linalg.lstsq(A, lt, rcond=None)
        if coef[2] < 0:                                       # concave in log space
            tstar = float(np.clip(-coef[1]/(2*coef[2]), -0.004, 0.004))   # trust radius
            r2 = 1 - np.sum((A@coef - lt)**2) / max(np.sum((lt-lt.mean())**2), 1e-12)
            out.append((np.clip(inc + tstar*u, 0, 1),
                        f"1-D log-quadratic vertex on {len(sel)} near-peak pts "
                        f"(t*={tstar:+.4f}, curvature a={coef[2]:.0f}, R2={r2:.2f})"))
    for xc in local_cloud(fid, r=0.003, n=200, df=df):
        out.append((xc, "micro TR cloud (r=0.003) around incumbent"))
    return out, dict(log=False)

# --- stochastic_capped (F2, F3): the lottery ------------------------------------
def strat_stochastic_capped(fid, df=None):
    X, y, _ = xy(fid, df); inc = X[np.argmax(y)]
    cands = [(np.clip(inc + 2e-3, 0, 1),
              "near-incumbent re-read (fresh noisy draw; exact repeat banned)")]
    cands += maximin_candidates(fid, n=3, df=df)
    for xc in local_cloud(fid, r=0.06, n=200, df=df):
        cands.append((xc, "near-incumbent lottery (high P(beat), small gain)"))
    return cands, dict(log=False)

# --- local_peak (F4; F6 post-W7): TuRBO-lite -------------------------------------
def strat_local_peak(fid, df=None):
    Xw, yw = weekly_trace(fid, df); X, y, _ = xy(fid, df); out = []
    if len(yw) >= 2:
        prev_best = np.max(y[np.arange(len(y)) != int(np.argmax(y))]) if yw[-1] == y.max() else y.max()
        succ = yw[-1] >= prev_best - 1e-12
        step = Xw[-1] - Xw[-2]
        f = 1.5 if succ else -0.5
        out.append((np.clip(Xw[-1] + f*step, 0, 1),
                    f"trust-region {'expand x1.5' if succ else 'reverse-halve'} along last step "
                    f"(|step|={np.linalg.norm(step):.3f})"))
    for xc in local_cloud(fid, r=0.04, n=300, df=df):
        out.append((xc, "TR cloud (r=0.04)"))
    return out, dict(log=False)

# --- boundary_climb (F5): coordinate isolation with capped extrapolation ---------
def strat_boundary_climb(fid, df=None):
    X, y, _ = xy(fid, df); d = DIMS[fid]; inc = X[np.argmax(y)]; out = []
    for j in range(d):
        others = np.delete(np.arange(d), j)
        online = np.abs(X[:, others] - inc[others]).max(axis=1) < 1e-9 if d > 1 else np.ones(len(X), bool)
        vals = X[online, j]
        if len(vals) >= 2 and vals.max() - vals.min() > 1e-9:
            span = vals.max() - vals.min()
            lo, hi = max(0, vals.min() - 2*span), min(1, vals.max() + 2*span)
            tag = f"(on-line span {span:.3f}, capped extrapolation)"
        else:
            lo, hi, tag = 0, 1, "(no on-line data — extrapolated, review before trusting)"
        for v in np.linspace(lo, hi, 41):
            c = inc.copy(); c[j] = v
            out.append((c, f"coord-scan x{j+1}={v:.3f} {tag}"))
    return out, dict(log=True)

# --- plateau (F6 weeks 4-6): price the escape hypotheses --------------------------
def strat_plateau(fid, df=None):
    X, y, _ = xy(fid, df); d = DIMS[fid]; inc = X[np.argmax(y)]; out = []
    cols = [np.ones((len(X), 1))]
    for j in range(d): cols += [X[:, [j]], X[:, [j]]**2, X[:, [j]]**3]
    Phi = np.hstack(cols); reg = Ridge(alpha=1.0).fit(Phi, y)
    grid = np.linspace(0, 1, 201); opt = inc.copy()
    for j in range(d):
        Xg = np.tile(inc, (201, 1)); Xg[:, j] = grid
        Pg = [np.ones((201, 1))]
        for jj in range(d): Pg += [Xg[:, [jj]], Xg[:, [jj]]**2, Xg[:, [jj]]**3]
        opt[j] = grid[np.argmax(reg.predict(np.hstack(Pg)))]
    out.append((opt, f"separable cubic coordinate-ascent optimum (in-sample R2={reg.score(Phi, y):.2f})"))
    for a in range(d):
        for b in range(a+1, d):
            for s in (+1, -1):
                c = inc.copy()
                c[a] = np.clip(c[a] + s*0.1, 0, 1); c[b] = np.clip(c[b] - s*0.1, 0, 1)
                out.append((c, f"anti-corr pair x{a+1}{'+' if s>0 else '-'}0.1 / x{b+1}{'-' if s>0 else '+'}0.1"))
    for j in range(d):
        for s in (+1, -1):
            c = inc.copy(); c[j] = np.clip(c[j] + s*0.12, 0, 1)
            out.append((c, f"single-factor x{j+1}{s:+d}x0.12 (clean attribution)"))
    out += maximin_candidates(fid, n=2, df=df)
    for xc in local_cloud(fid, r=0.05, n=150, df=df):
        out.append((xc, "local jitter"))
    return out, dict(log=False)

# --- solved_lock (F7, F8): epsilon on the least-sensitive dim ----------------------
def strat_solved_lock(fid, df=None):
    X, y, _ = xy(fid, df); inc = X[np.argmax(y)]
    x, note = epsilon_fix(fid, inc, df=df)
    return [(x, note or "epsilon lock")], dict(skip_referee=True)

STRATS = dict(needle=strat_needle, stochastic_capped=strat_stochastic_capped,
              local_peak=strat_local_peak, boundary_climb=strat_boundary_climb,
              plateau=strat_plateau, solved_lock=strat_solved_lock)

## 5 · Run: classify → generate → price → pick

In [6]:
def suggest(fid, top=6, df=None):
    label, ev = classify(fid, df)
    cands, opts = STRATS[label](fid, df=df)
    if opts.get("skip_referee"):
        x = cands[0][0]
        tbl = pd.DataFrame([dict(EI=0.0, P_beat=0.0, provenance=cands[0][1],
                                 x=tuple(np.round(x, 6)))])
        return label, ev, tbl, x
    gp, ym = fit_ref(fid, log=opts.get("log", False), df=df)
    XC = np.array([c[0] for c in cands]); prov = [c[1] for c in cands]
    ei, pb, mu, s = price(gp, ym.max(), XC)
    tbl = pd.DataFrame(dict(EI=ei, P_beat=pb, mu=mu, sd=s, provenance=prov))
    tbl["x"] = [tuple(np.round(v, 6)) for v in XC]
    tbl = tbl.sort_values("EI", ascending=False).head(top).reset_index(drop=True)
    xpick = np.array(tbl.loc[0, "x"], float)
    xpick, note = epsilon_fix(fid, xpick, df=df)
    if note: tbl.loc[0, "provenance"] += " | " + note
    return label, ev, tbl, xpick

RESULTS = {}
for f in range(1, 9):
    label, ev, tbl, x = suggest(f)
    RESULTS[f] = dict(label=label, evidence="; ".join(ev), table=tbl, pick=x)
    print(f"\n=== F{f} -> {label} ===")
    print("   evidence:", RESULTS[f]["evidence"])
    print(tbl[["EI", "P_beat", "provenance", "x"]].to_string(index=False))


=== F1 -> local_peak ===
   evidence: new best-ever last week (+0) via a 0.000-norm step — verified local gradient
      EI   P_beat        provenance                    x
0.006298 0.030779 TR cloud (r=0.04) (0.644156, 0.601884)
0.006289 0.029329 TR cloud (r=0.04) (0.653502, 0.605596)
0.006265 0.029611 TR cloud (r=0.04) (0.655213, 0.609005)
0.006257 0.030959 TR cloud (r=0.04) (0.641292, 0.601159)
0.006228 0.031502 TR cloud (r=0.04)  (0.643534, 0.60342)
0.006188 0.030487 TR cloud (r=0.04) (0.654386, 0.611859)

=== F2 -> plateau ===
   evidence: no new best for 3 weeks; incumbent 0.6505
      EI   P_beat                     provenance                    x
0.036677 0.358350 anti-corr pair x1+0.1 / x2-0.1     (0.7854, 0.2325)
0.035233 0.431447                   local jitter (0.731382, 0.354395)
0.035227 0.345096                   local jitter (0.789012, 0.241023)
0.035209 0.418587                   local jitter  (0.725223, 0.36697)
0.035188 0.427404                   local jitter (0.72661


=== F3 -> local_peak ===
   evidence: default: learnable, not stalled
      EI   P_beat        provenance                              x
0.020402 0.384442 TR cloud (r=0.04) (0.306054, 0.540484, 0.529115)
0.019858 0.392142 TR cloud (r=0.04) (0.503032, 0.446494, 0.506498)
0.018963 0.373159 TR cloud (r=0.04) (0.320659, 0.557854, 0.527017)
0.018855 0.390020 TR cloud (r=0.04)  (0.476907, 0.47834, 0.532585)
0.018447 0.376784 TR cloud (r=0.04) (0.388365, 0.541468, 0.559845)
0.018404 0.384629 TR cloud (r=0.04) (0.375737, 0.533444, 0.555537)



=== F4 -> local_peak ===
   evidence: new best-ever last week (+0.0147) via a 0.007-norm step — verified local gradient
      EI   P_beat        provenance                                        x
0.173045 0.589722 TR cloud (r=0.04) (0.439084, 0.421578, 0.306838, 0.465977)
0.146767 0.559509 TR cloud (r=0.04) (0.431868, 0.397236, 0.320766, 0.459121)
0.121682 0.551824 TR cloud (r=0.04) (0.418871, 0.397677, 0.330357, 0.452558)
0.110796 0.525688 TR cloud (r=0.04) (0.442858, 0.450093, 0.325519, 0.439505)
0.105073 0.461303 TR cloud (r=0.04) (0.410938, 0.401263, 0.333741, 0.468835)
0.098993 0.522824 TR cloud (r=0.04) (0.444351, 0.422011, 0.310658, 0.417873)



=== F5 -> boundary_climb ===
   evidence: 4/4 coords at bounds; mean |rho|=0.60; huge monotone range
      EI   P_beat                                                                   provenance                      x
0.012691 0.050472 coord-scan x4=0.800 (no on-line data — extrapolated, review before trusting)   (1.0, 1.0, 1.0, 0.8)
0.012634 0.046143 coord-scan x4=0.775 (no on-line data — extrapolated, review before trusting) (1.0, 1.0, 1.0, 0.775)
0.012518 0.055118 coord-scan x4=0.825 (no on-line data — extrapolated, review before trusting) (1.0, 1.0, 1.0, 0.825)
0.012396 0.042128 coord-scan x4=0.750 (no on-line data — extrapolated, review before trusting)  (1.0, 1.0, 1.0, 0.75)
0.012060 0.060080 coord-scan x4=0.850 (no on-line data — extrapolated, review before trusting)  (1.0, 1.0, 1.0, 0.85)
0.012018 0.038418 coord-scan x4=0.725 (no on-line data — extrapolated, review before trusting) (1.0, 1.0, 1.0, 0.725)



=== F6 -> plateau ===
   evidence: no new best for 3 weeks; incumbent -0.2347
      EI   P_beat                                  provenance                                                 x
0.013210 0.218781                                local jitter     (0.246126, 0.321303, 0.612008, 0.717899, 0.0)
0.013024 0.249544                                local jitter     (0.297743, 0.321828, 0.620098, 0.685495, 0.0)
0.012699 0.247512                                local jitter     (0.280271, 0.351628, 0.652925, 0.652081, 0.0)
0.012508 0.248743                                local jitter   (0.270133, 0.3541, 0.692505, 0.67517, 0.001237)
0.011531 0.277818                                local jitter (0.30613, 0.383007, 0.668422, 0.686655, 0.005481)
0.010916 0.263336 single-factor x2-1x0.12 (clean attribution)                     (0.35, 0.32, 0.66, 0.71, 0.0)

=== F7 -> solved_lock ===
   evidence: analytic identity confirmed (human prior); optimum banked
 EI  P_beat                            

In [7]:
summary = pd.DataFrame([{
    "F": f"F{f}", "pathology": r["label"], "evidence": r["evidence"],
    "engine pick": "-".join(f"{v:.6f}" for v in r["pick"]),
    "EI": round(float(r["table"].loc[0, "EI"]), 4),
    "P(beat)": round(float(r["table"].loc[0, "P_beat"]), 2),
} for f, r in RESULTS.items()])
pd.set_option("display.max_colwidth", 95)
summary

,F,pathology,evidence,engine pick,EI,P(beat)
0,F1,local_peak,new best-ever last week (+0) via a 0.000-norm step — verified local gradient,0.644156-0.601884,0.0063,0.03
1,F2,plateau,no new best for 3 weeks; incumbent 0.6505,0.785400-0.232500,0.0367,0.36
2,F3,local_peak,"default: learnable, not stalled",0.306054-0.540484-0.529115,0.0204,0.38
3,F4,local_peak,new best-ever last week (+0.0147) via a 0.007-norm step — verified local gradient,0.439084-0.421578-0.306838-0.465977,0.1730,0.59
4,F5,boundary_climb,4/4 coords at bounds; mean |rho|=0.60; huge monotone range,1.000000-1.000000-1.000000-0.800000,0.0127,0.05
5,F6,plateau,no new best for 3 weeks; incumbent -0.2347,0.246126-0.321303-0.612008-0.717899-0.000000,0.0132,0.22
6,F7,solved_lock,analytic identity confirmed (human prior); optimum banked,0.201689-0.150011-0.476874-0.275333-0.311652-0.657301,0.0000,0.00
7,F8,solved_lock,analytic identity confirmed (human prior); optimum banked,0.100000-0.150000-0.130000-0.150000-0.800000-0.500001-0.200000-0.600000,0.0000,0.00


## 6 · Submission builder — engine picks + logged human overrides

Overrides live here, exactly like the campaign's F2/F3/F6 calls: the engine's referee-ranked pick
stays in the log, the human choice replaces it, and the comparison is itself a finding when the
results come back.

In [8]:
# HUMAN_OVERRIDES[f] = (np.array([...]), "reason")  — leave empty to submit the engine picks.
HUMAN_OVERRIDES = {}

def portal_string(x):
    return "-".join(f"{v:.6f}" for v in np.clip(np.asarray(x, float), 0, 1))

final, provlog = {}, []
for f in range(1, 9):
    if f in HUMAN_OVERRIDES:
        x, why = HUMAN_OVERRIDES[f]
        x, note = epsilon_fix(f, x)
        final[f] = x
        provlog.append(dict(F=f"F{f}", provenance=f"HUMAN OVERRIDE — {why}"
                            + (f" | {note}" if note else ""),
                            engine_alt=portal_string(RESULTS[f]["pick"])))
    else:
        final[f] = RESULTS[f]["pick"]
        provlog.append(dict(F=f"F{f}",
                            provenance=f"engine ({RESULTS[f]['label']}): "
                                       + RESULTS[f]["table"].loc[0, "provenance"],
                            engine_alt=""))

official = "|".join(portal_string(final[f]) for f in range(1, 9))
print(f"=== Week {N_WEEKS+1} submission ===\n")
for f in range(1, 9):
    print(f"  F{f}: {portal_string(final[f])}")
print("\nCombined:\n" + official)
pd.DataFrame(provlog)

=== Week 14 submission ===

  F1: 0.644156-0.601884
  F2: 0.785400-0.232500
  F3: 0.306054-0.540484-0.529115
  F4: 0.439084-0.421578-0.306838-0.465977
  F5: 1.000000-1.000000-1.000000-0.800000
  F6: 0.246126-0.321303-0.612008-0.717899-0.000000
  F7: 0.201689-0.150011-0.476874-0.275333-0.311652-0.657301
  F8: 0.100000-0.150000-0.130000-0.150000-0.800000-0.500001-0.200000-0.600000

Combined:
0.644156-0.601884|0.785400-0.232500|0.306054-0.540484-0.529115|0.439084-0.421578-0.306838-0.465977|1.000000-1.000000-1.000000-0.800000|0.246126-0.321303-0.612008-0.717899-0.000000|0.201689-0.150011-0.476874-0.275333-0.311652-0.657301|0.100000-0.150000-0.130000-0.150000-0.800000-0.500001-0.200000-0.600000


,F,provenance,engine_alt
0,F1,engine (local_peak): TR cloud (r=0.04),
1,F2,engine (plateau): anti-corr pair x1+0.1 / x2-0.1,
2,F3,engine (local_peak): TR cloud (r=0.04),
3,F4,engine (local_peak): TR cloud (r=0.04),
4,F5,"engine (boundary_climb): coord-scan x4=0.800 (no on-line data — extrapolated, review before...",
5,F6,engine (plateau): local jitter,
6,F7,engine (solved_lock): repeat detected -> eps +1e-06 on x4 (least-sensitive),
7,F8,engine (solved_lock): repeat detected -> eps +1e-06 on x6 (least-sensitive),


## 7 · Backtest — does the classifier track the campaign's own history?

Re-run the classifier on the data truncated at each past week. The labels should *move* the way the
campaign's understanding moved (that is the accuracy test that matters for a router): F6 plateau →
`local_peak` when the week-7 probe landed, F4 settling into `local_peak` once the gradient was
verified, F1 needle from the start.

In [9]:
hist = []
for w in range(2, N_WEEKS + 1):
    dfw = OBS[OBS.week <= w]
    row = {"through week": w}
    for f in range(1, 9):
        try:
            row[f"F{f}"] = classify(f, dfw)[0]
        except Exception as e:
            row[f"F{f}"] = f"err: {e}"
    hist.append(row)
pd.DataFrame(hist).set_index("through week")

,F1,F2,F3,F4,F5,F6,F7,F8
through week,,,,,,,,
2,needle,local_peak,local_peak,local_peak,local_peak,local_peak,solved_lock,solved_lock
3,needle,stochastic_capped,local_peak,local_peak,local_peak,local_peak,solved_lock,solved_lock
4,needle,stochastic_capped,stochastic_capped,local_peak,local_peak,local_peak,solved_lock,solved_lock
5,needle,stochastic_capped,stochastic_capped,local_peak,local_peak,local_peak,solved_lock,solved_lock
6,needle,stochastic_capped,stochastic_capped,local_peak,boundary_climb,plateau,solved_lock,solved_lock
7,needle,stochastic_capped,stochastic_capped,local_peak,boundary_climb,local_peak,solved_lock,solved_lock
8,needle,stochastic_capped,stochastic_capped,local_peak,boundary_climb,local_peak,solved_lock,solved_lock
9,needle,local_peak,local_peak,local_peak,boundary_climb,local_peak,solved_lock,solved_lock
10,needle,local_peak,local_peak,local_peak,boundary_climb,local_peak,solved_lock,solved_lock


## 8 · Export

In [10]:
with open(f"week{N_WEEKS+1}_engine_submission.txt", "w") as fh:
    fh.write(official + "\n")
summary.to_csv("engine_summary.csv", index=False)
pd.DataFrame(provlog).to_csv("engine_provenance.csv", index=False)
print("Wrote: week%d_engine_submission.txt, engine_summary.csv, engine_provenance.csv" % (N_WEEKS+1))
print("\nWeekly loop: submit -> append returned rows -> re-run 04 then this notebook.")
print("The classifier re-labels every function each week; overrides go in PATHOLOGY_OVERRIDES")
print("(archetype) or HUMAN_OVERRIDES (query), both logged.")

Wrote: week14_engine_submission.txt, engine_summary.csv, engine_provenance.csv

Weekly loop: submit -> append returned rows -> re-run 04 then this notebook.
The classifier re-labels every function each week; overrides go in PATHOLOGY_OVERRIDES
(archetype) or HUMAN_OVERRIDES (query), both logged.
